In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install datarobot

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 708.5/708.5 kB 11.6 MB/s eta 0:00:00


In [3]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.1 MB/s eta 0:00:00


In [4]:
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=1e125b535097c715ddd9a774b91cfd3a11c89963b80b041ff2ba6f7201ac3606
  Stored in directory: /root/.cache/pip/wheels/bc/92/f0/243288f899c2eacdfa8c5f9aede4c71a9bad0ee26a01dc5ead
Successfully built seqeval


In [5]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline, AutoModelForCausalLM
import datarobot as dr
import json
import os
os.environ["WANDB_DISABLED"] = "true"
from datasets import Dataset
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForTokenClassification

In [6]:
import numpy as np
import evaluate
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

metric = evaluate.load("seqeval")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [7]:
# Use this for 1L sentences
folder_path = '/content/drive/MyDrive/Konkani_ILCI_POS_Tagged_1Lakh_Sentences'
files = os.listdir(folder_path)
txt_files = [file for file in files if file.endswith('.txt')]
len(txt_files)

100

In [8]:
if "kon_agriculture_set1.txt" in txt_files:
    txt_files.remove("kon_agriculture_set1.txt")

print(len(txt_files))

99


In [9]:
taggedData = []
for file in txt_files:
    with open("/content/drive/MyDrive/Konkani_ILCI_POS_Tagged_1Lakh_Sentences/" + file, 'r', encoding='utf-8') as file:
        content = file.read()
    content = content.split("\n")
    content = content[1:]  # Skip the first line
    taggedDataFile = [line.split('\t')[1] for line in content if 'Value' not in line.split('\t')[1]]
    taggedData.append(taggedDataFile)

allTaggedData = [item for sublist in taggedData for item in sublist]
allTaggedData[:2]

['इस्वी\\N_NN वर्स\\N_NN १९६७त\\N_NN आमच्या\\PR_PRP विज्ञानिकांनी\\N_NN कापसाचे\\N_NN उत्पन्न\\N_NN वाडोवपा\\V_VM_VNF खातीर\\PSP एका\\QT_QTC व्हड\\JJ कार्यक्रमाक\\N_NN सुरवात\\N_NN केली\\V_VM_VF जाका\\DM_DMR लागून\\PSP थोड्याच\\QT_QTF वर्सा\\N_NN भितर\\PSP प्रती\\N_NN हॅक्टर\\N_NN उत्पादन\\N_NN आनी\\CC_CCD पुराय\\JJ उत्पादन\\N_NN दोनाय\\QT_QTC मदीं\\PSP दुपेटीच्यान\\N_NN चड\\QT_QTF वाड\\N_NN जालीं\\V_VM_VF .\\RD_PUNC',
 'भारतांत\\N_NNP कापसाचे\\N_NN शेतीचे\\N_NN क्षेत्र\\N_NN संवसारांत\\N_NN सगळ्यांत\\RP_INTF चड\\QT_QTF (\\RD_PUNC सुमार\\RB ८०\\QT_QTC लाख\\N_NN हॅक्टर\\N_NN )\\RD_PUNC आसा\\V_VM_VF आनी\\CC_CCD होच\\DM_DMD एकमेव\\N_NN असो\\DM_DMD देश\\N_NN आसा\\V_VM_VF जंय\\N_NST कापसाची\\N_NN कृषी\\N_NN योग्य\\JJ जातींची\\N_NN शेती\\N_NN करतात\\V_VM_VF .\\RD_PUNC']

In [10]:
df = pd.DataFrame(allTaggedData, columns=['sentences_pos_tagged'])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99091 entries, 0 to 99090
Data columns (total 1 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   sentences_pos_tagged  99091 non-null  object
dtypes: object(1)
memory usage: 774.3+ KB


In [11]:
df.head()

,sentences_pos_tagged
0,इस्वी\N_NN वर्स\N_NN १९६७त\N_NN आमच्या\PR_PRP ...
1,भारतांत\N_NNP कापसाचे\N_NN शेतीचे\N_NN क्षेत्र...
2,पूण\CC_CCS उत्पादनांत\N_NN भारताचें\N_NNP स्था...
3,भारतांत\N_NNP कापसाच्या\N_NN उत्पादनांत\N_NN उ...
4,राज्यांत\N_NN प्रती\N_NN एकर\N_NN उत्पादनाच्या...


In [12]:
def preprocess_sentence(sentence):
    words, tags = [], []
    for token in sentence.split():
        if "\\" in token:
            word, tag = token.rsplit("\\", 1)  # Split at last occurrence of '\'
            words.append(word)
            tags.append(tag)
    return words, tags

# Apply preprocessing to extract words & POS tags
df["tokens"], df["labels"] = zip(*df["sentences_pos_tagged"].map(preprocess_sentence))

In [13]:
df.head()

,sentences_pos_tagged,tokens,labels
0,इस्वी\N_NN वर्स\N_NN १९६७त\N_NN आमच्या\PR_PRP ...,"[इस्वी, वर्स, १९६७त, आमच्या, विज्ञानिकांनी, का...","[N_NN, N_NN, N_NN, PR_PRP, N_NN, N_NN, N_NN, V..."
1,भारतांत\N_NNP कापसाचे\N_NN शेतीचे\N_NN क्षेत्र...,"[भारतांत, कापसाचे, शेतीचे, क्षेत्र, संवसारांत,...","[N_NNP, N_NN, N_NN, N_NN, N_NN, RP_INTF, QT_QT..."
2,पूण\CC_CCS उत्पादनांत\N_NN भारताचें\N_NNP स्था...,"[पूण, उत्पादनांत, भारताचें, स्थान, अमेरिका, ,,...","[CC_CCS, N_NN, N_NNP, N_NN, N_NNP, RD_PUNC, N_..."
3,भारतांत\N_NNP कापसाच्या\N_NN उत्पादनांत\N_NN उ...,"[भारतांत, कापसाच्या, उत्पादनांत, उणाव, जावपाचे...","[N_NNP, N_NN, N_NN, N_NN, V_VM_VNF, JJ, N_NN, ..."
4,राज्यांत\N_NN प्रती\N_NN एकर\N_NN उत्पादनाच्या...,"[राज्यांत, प्रती, एकर, उत्पादनाच्या, हिशोबान, ...","[N_NN, N_NN, N_NN, N_NN, N_NN, N_NNP, RP_INTF,..."


In [14]:
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.1)

print("Train Length: ",len(dataset["train"]))
print("Test Length: ",len(dataset["test"]))
print("Train first ex: ",dataset["train"][0])

Train Length:  89181
Test Length:  9910
Train first ex:  {'sentences_pos_tagged': 'भारतीय\\JJ -\\RD_PUNC आर्य\\JJ लोकां\\N_NN वांगडा\\PSP ब्राह्मी\\N_NNP लिपयेचो\\N_NN एक\\QT_QTC फांटो\\N_NN श्रीलंकेकूय\\N_NNP पावलो\\V_VM_VF .\\RD_PUNC', 'tokens': ['भारतीय', '-', 'आर्य', 'लोकां', 'वांगडा', 'ब्राह्मी', 'लिपयेचो', 'एक', 'फांटो', 'श्रीलंकेकूय', 'पावलो', '.'], 'labels': ['JJ', 'RD_PUNC', 'JJ', 'N_NN', 'PSP', 'N_NNP', 'N_NN', 'QT_QTC', 'N_NN', 'N_NNP', 'V_VM_VF', 'RD_PUNC']}


In [15]:
model_name = "xlm-roberta-base"
#model_name = "muhammadbilal/xlm-roberta-base-finetuned-pos" #Done
#model_name = "ai4bharat/indic-bert" #Done
#model_name = "google/muril-base-cased"
#model_name = "google/muril-large-cased" #Done
#model_name = "google-bert/bert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Define POS tags
label_list = list(set(tag for tags in df["labels"] for tag in tags))
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

# Load pre-trained model for token classification
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


XLMRobertaForTokenClassification(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768

In [16]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], is_split_into_words=True, padding="max_length", truncation=True)

    labels = []
    for i, label in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        previous_word_idx = None
        for word_idx in word_ids:
            if word_idx is None or word_idx == previous_word_idx:
                label_ids.append(-100)
            else:
                label_ids.append(label2id[label[word_idx]])
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Apply tokenization
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/89181 [00:00<?, ? examples/s]

Map:   0%|          | 0/9910 [00:00<?, ? examples/s]

In [17]:
training_args = TrainingArguments(
    output_dir="./konkani_pos_model",
    run_name="konkani_pos_tagger",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-05,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs",
    report_to="none",
    fp16=True,
)

In [18]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_labels = [[label for label in label_seq if label != -100] for label_seq in labels]
    pred_labels = [[label for label, true in zip(pred_seq, label_seq) if true != -100] for pred_seq, label_seq in zip(predictions, labels)]

    accuracy = accuracy_score(sum(true_labels, []), sum(pred_labels, []))
    precision, recall, f1, _ = precision_recall_fscore_support(sum(true_labels, []), sum(pred_labels, []), average='weighted')
    class_report = classification_report(sum(true_labels, []), sum(pred_labels, []))

    print("Classification Report:\n", class_report)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss


In [ ]:
# Evaluate the Model
results = trainer.evaluate()
print(results)

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Classification Report:
               precision    recall  f1-score   support

           3       0.43      0.56      0.48       600
           4       0.00      0.00      0.00         1
           5       0.85      0.85      0.85      8501
           7       0.93      0.77      0.84       482
           8       0.92      0.94      0.93       570
           9       0.00      0.00      0.00         9
          10       0.56      0.28      0.37       496
          11       0.00      0.00      0.00         1
          12       0.85      0.91      0.88      2851
          14       0.00      0.00      0.00        11
          15       0.00      0.00      0.00         6
          16       0.90      0.92      0.91      4956
          18       0.53      0.95      0.68       663
          20       0.77      0.76      0.76       469
          21       0.00      0.00      0.00       102
          26       0.61      0.90      0.73       359
          28       0.00      0.00      0.00         2
   

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
model.save_pretrained("./konkani_ai4bharat_indic-bert")
tokenizer.save_pretrained("./konkani_ai4bharat_indic-bert")

('./konkani_ai4bharat_indic-bert/tokenizer_config.json',
 './konkani_ai4bharat_indic-bert/special_tokens_map.json',
 './konkani_ai4bharat_indic-bert/spiece.model',
 './konkani_ai4bharat_indic-bert/added_tokens.json',
 './konkani_ai4bharat_indic-bert/tokenizer.json')

In [ ]:
from transformers import pipeline

# Load model
pos_tagger = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="first"
)

# Sample Konkani sentence
#sentence = "ग्रेटर नोयडा वेस्टाचे त्रास कमी जावपाचें नांवूच घेना  ."
sentence = " ".join(dataset["test"][0]["tokens"])
print("Input Sentence: ", sentence)

words = sentence.split()

# Perform POS tagging
output = pos_tagger(words)

flattened_output = [item[0] for item in output]
formatted_output = " ".join([f"{entry['word']}\\{entry['entity_group']}" for entry in flattened_output])
print("POS Tagged Sentence: ", dataset["test"][0]["sentences_pos_tagged"])
print("Predicted Output POS Tagged Sentence: ",formatted_output)

Device set to use cuda:0
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Input Sentence:  औरंगाबाद दक्षीण मध्य रेल्वेचें म्हत्त्वाचें जंक्शन .
POS Tagged Sentence:  औरंगाबाद\N_NNP दक्षीण\N_NNP मध्य\JJ रेल्वेचें\N_NN म्हत्त्वाचें\JJ जंक्शन\N_NN .\RD_PUNC
Predicted Output POS Tagged Sentence:  औरगबद\N_NNP दकषण\N_NNP मधय\N_NN रलवच\N_NN महततवच\N_NN जकशन\N_NN .\RD_PUNC


In [ ]:
# Load model
pos_tagger = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="first"
)

def predict(sentence):
    words = sentence.split()
    output = pos_tagger(words)
    flattened_output = [item[0] for item in output]
    formatted_output = " ".join([f"{entry['word']}\\{entry['entity_group']}" for entry in flattened_output])
    return formatted_output

predict("आयकलां ? लागीं ना आमचो गांव . चलून चलून कितलीं म्हणून चलतलीं आमी ?")

In [ ]:
import pandas as pd
df_ref = pd.read_excel('/content/anwani_ref_sentences.xlsx', engine='openpyxl')
df_ref.head()

In [ ]:
df_ref["output"] = df_ref["sentences_cleaned"].apply(predict)
df_ref.head()

In [ ]:
import pandas as pd

def add_length_columns(df):
    # Compute lengths of actual_tags and predicted_sentences
    df['ref_sentences_length'] = df['tagged_sentences'].apply(lambda x: len(str(x).split()))
    df['joined_sent_length'] = df['output'].apply(lambda x: len(str(x).split()))

    # Find row indices where lengths do not match
    mismatched_rows = df[df['ref_sentences_length'] != df['joined_sent_length']].index.tolist()

    # Remove mismatched rows from the DataFrame
    #df = df.drop(mismatched_rows)

    return df, mismatched_rows

# Example usage:
df_ref, mismatched_indices = add_length_columns(df_ref)
print(mismatched_indices)

for ind in mismatched_indices:
    df_ref = df_ref.drop(ind)

df_ref.info()

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

def evaluate_tagging(ref_sent, predicted_sent):
    #print("ref_sent: ", ref_sent)
    #print("predicted_sent: ", predicted_sent)
    # Handle missing values
    if not isinstance(ref_sent, str) or not isinstance(predicted_sent, str):
        return 0.0, 0.0, 0.0, 0.0  # Return default scores for missing values

    # Split sentences into word-tag pairs
    ref_tags = ref_sent.split()
    pred_tags = predicted_sent.split()

    if len(ref_tags) != len(pred_tags):
        return 0.0, 0.0, 0.0, 0.0


    # Extract only the tags from word\tag pairs
    ref_labels = [token.split("\\")[-1] for token in ref_tags]
    pred_labels = [token.split("\\")[-1] for token in pred_tags]

    #print(len(ref_labels), len(pred_labels))

    # Compute accuracy
    accuracy = accuracy_score(ref_labels, pred_labels)

    # Compute precision, recall, and F1-score (weighted for class imbalance)
    precision, recall, f1, _ = precision_recall_fscore_support(ref_labels, pred_labels, average='weighted', zero_division=0)

    return accuracy, precision, recall, f1, ref_labels, pred_labels

In [ ]:
# Apply function to each row
df_ref[["Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "Ref Labels",
        "Pred Labels"]] = df_ref.apply(lambda row: evaluate_tagging(row["tagged_sentences"], row["output"]),
                                       axis=1,
                                       result_type="expand")
df_ref.head()

In [ ]:
# Flatten all reference and predicted labels for classification report
all_ref_labels = [label for labels in df_ref["Ref Labels"] for label in labels]
all_pred_labels = [label for labels in df_ref["Pred Labels"] for label in labels]

# Generate classification report
print("\nClassification Report:\n")
print(classification_report(all_ref_labels, all_pred_labels, zero_division=0))

In [ ]:
df_ref.to_excel("test_xlmr-base-finetuned-pos_100_anwani_output.xlsx", index=False)